In [1]:
import nemo.collections.asr as nemo_asr

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
Skipping import of cpp extensions due to incompatible torch version 2.8.0a0+5228986c39.nv25.06 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
[NeMo W 2025-12-03 18:59:00 nemo_logging:405] /opt/venv/lib/python3.12/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
      warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", Run

In [2]:
import torch
device='cuda' if torch.cuda.is_available() else 'cpu'
print (device)

cuda


In [3]:
asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v3")


#updating self-attention model of fast-conformer encoder
#setting attention left and right context sizes to 256
asr_model.change_attention_model(self_attention_model="rel_pos_local_attn", att_context_size=[256, 256])



[NeMo I 2025-12-03 19:02:18 nemo_logging:393] Tokenizer SentencePieceTokenizer initialized with 8192 tokens


[NeMo W 2025-12-03 19:02:21 nemo_logging:405] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2025-12-03 19:02:21 nemo_logging:405] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    V

[NeMo I 2025-12-03 19:02:21 nemo_logging:393] PADDING: 0
[NeMo I 2025-12-03 19:02:26 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2025-12-03 19:02:26 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2025-12-03 19:02:26 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2025-12-03 19:02:28 nemo_logging:393] Model EncDecRNNTBPEModel was successfully restored from /home/896166/.cache/huggingface/hub/models--nvidia--parakeet-tdt-0.6b-v3/snapshots/6d590f77001d318fb17a0b5bf7ee329a91b52598/parakeet-tdt-0.6b-v3.nemo.


In [10]:
audio_file = 'whisper/real_case/CaseStudies2.WAV'
audio_id = audio_file.replace('.wav','')
#print (audio_id)
audio_id = to_16k_mono_audio(audio_id)
output = asr_model.transcribe([audio_id], timestamps=True)
#segment_time = output[0].timestamp['segment']

print (output[0].text)
'''for stamp in segment_time:
    print (stamp['start'])
    print (stamp['end'])
    print (stamp['segment'])
print(output[0].text)'''


with open('whisper/real_case/parakeet_case2.txt', 'w') as f:
    f.write(output[0].text)

[NeMo I 2025-12-03 19:18:49 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2025-12-03 19:18:49 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


Transcribing: 100%|██████████| 1/1 [00:15<00:00, 15.52s/it]

So we're discussing this patient to make a treatment plan for her. She is a lady who has presented to us with a lump in the upper central right breast. We have imaged her and on the ultrasound scan it measures 16 millimeters, mammograms 22 millimeters, and we've done a core biopsy that shows a grade 2 invasive lobular carcinoma that's ER6PR2HER2 fish negative, and the biopsy from the axilla was normal. She has been discussed at the MDT before. Now she's previously had a right white local excision for 5 mm of high grade DCIS, and she had radiotherapy after surgery in 2009. The MDT recommendation was for a right mistectomy in a center of node biopsy. Patient declined a mistectomy, and so the question was what do we do next? She wants to have breast-conserving surgery and she wants a lumpectomy. There are some places who are offering redo wide local excisions and postoperative radiotherapy. This patient had a really poor experience from her previous radiotherapy and only wants a wide loca

In [6]:
output

[Hypothesis(score=-725.4371948242188, y_sequence=tensor([1030, 3127,  298, 1605,  508,  356, 6832, 7883, 6373,  292, 7877, 7863,
          239,  240, 7877,  278, 1413,  326, 1840,  431,  603,  351,  521, 1633,
          306,  274,  381, 7883, 6474,  663, 7863,  242, 3700,  850,  454,  334,
          506, 6332, 2380,  277, 2009,  277,  363,  319, 7871,  527, 7877,  818,
         2249, 2656, 1740,  297, 1472,  560, 1075,  787, 1743, 4387, 2015,  652,
          518, 7924, 7942, 7923, 7863,  239, 7883,  417,  916, 4149, 1447, 7893,
         4024, 2589, 7870, 1740,  297, 1472,  560, 1075,  787, 1743, 4387, 2015,
         7883, 6195,  401, 7863,  235, 7877,  518, 7778, 1472, 7877, 1535, 7863,
          236, 2104, 5659, 7883, 1169,  314, 3720,  786, 4757, 4347, 6286,  313,
         7893,  298, 7550,  580, 7883, 1245, 4322,  295, 1740,  324, 7879,  926,
          431, 7032,  381, 7883, 1960, 7877, 1637,  306,  326, 7877,  328, 1267,
         3224,  295,  743, 2018,  454, 7877,  575, 5044,  281

In [6]:
import numpy as np
def to_16k_mono_audio(file: str) -> np.ndarray:
    audio_arr, sr = librosa.load(file, sr= None)
    
    if len(audio_arr.shape)>1:
        audio_arr = librosa.to_mono(audio_arr)
        
    if sr!=16000:
        audio_arr = librosa.resample(audio_arr, orig_sr =sr, target_sr =16000)

    return audio_arr   


In [8]:
import librosa

In [4]:
import glob
import pandas as pd

import os
from tqdm import tqdm
from whisper_normalizer.english import EnglishTextNormalizer

scripts = glob.glob('whisper/augmented_audio/MDT_meeting_full_case9*')
normalizer = EnglishTextNormalizer()



for script in tqdm(scripts):
    print ('============================')
    #print (script)
    case =script.split("/")[-1]
    #print (case)
    audio_path = glob.glob(script+str('/*'))
    #print (audio_path[:4])
    

    if not os.path.exists(f'whisper/outputs/parakeet-viper/'):
        os.makedirs(f'whisper/outputs/parakeet-viper/')


    f = open(f"whisper/simulated_script/{case}.txt")
    
    
    
    case_script = f.read()
    #print (case_script)
    hypotheses =[]
    references= []
    file_ids = []
    trans_ex_all = []
    
    for audio_file in tqdm(audio_path):
        
        
        #print (audio_file)
        audio_id = audio_file.split("/")[-1].replace('.wav','')
        #print (audio_id)
        audio_file = to_16k_mono_audio(audio_file)
        result = asr_model.transcribe([audio_file], timestamps=True)
        segment_time = result[0].timestamp['segment']

        '''for stamp in segment_time:
            print (stamp['start'])
            print (stamp['end'])
            print (stamp['segment'])'''


        #result = model.transcribe(audio_file,fp16=False, language='en')
        
        for segment in segment_time:
            #print (segment)
            transc_extra ={}
            transc_extra = {'file_ids': audio_id}
            transc_extra =  transc_extra | segment
            trans_ex_all.extend([transc_extra])
            
        #print (transc_extra)
        
        transc = result[0].text

        #print (transc)
        
        hypotheses.extend([transc])
        references.extend([case_script])
        file_ids.extend([audio_id])
        
        

        #print (hypotheses)
        #print (references)
        #print (file_ids)

    data = pd.DataFrame(dict(hypothesis=hypotheses, reference=references, file_ids= file_ids))
    data["hypothesis_clean"] = [normalizer(text) for text in data["hypothesis"]]
    data["reference_clean"] = [normalizer(text) for text in data["reference"]]
    data2 = pd.DataFrame(trans_ex_all)
    
    
    data = data.merge(data2, on= 'file_ids', how='outer')
    
    
    #print (data)
    #print (data.columns)

    '''# Apply row-wise WER calculation
    data["WER"] = data.apply(
        lambda row: jiwer.wer(row["reference_clean"], row["hypothesis_clean"]),
        axis=1
    )
    data["WIL"] = data.apply(
        lambda row: jiwer.wil(row["reference_clean"], row["hypothesis_clean"]),
        axis=1
    )

    
    print("Average WER: {:.2f}%".format(data["WER"].mean() * 100))

   
    print("Average WER: {:.2f}%".format(data["WIL"].mean() * 100))'''
    
    
    data.to_csv(f'whisper/outputs/parakeet-viper/{case}.csv')
    #data2.to_csv(f'output/whisper-viper/extra_{case}.csv')

  0%|          | 0/1 [00:00<?, ?it/s]


  0%|          | 0/127 [00:00<?, ?it/s]

[NeMo I 2025-12-01 13:07:55 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2025-12-01 13:07:55 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}




Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing: 100%|██████████| 1/1 [00:11<00:00, 11.78s/it]

  1%|          | 1/127 [00:18<38:23, 18.28s/it]

[NeMo I 2025-12-01 13:08:07 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2025-12-01 13:08:07 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}


...



Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.32it/s]

 98%|█████████▊| 125/127 [02:13<00:01,  1.13it/s]

[NeMo I 2025-12-01 13:10:03 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2025-12-01 13:10:03 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}




Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]

 99%|█████████▉| 126/127 [02:14<00:00,  1.14it/s]

[NeMo I 2025-12-01 13:10:03 nemo_logging:393] Timestamps requested, setting decoding timestamps to True. Capture them in Hypothesis object,                         with output[0][idx].timestep['word'/'segment'/'char']
[NeMo I 2025-12-01 13:10:03 nemo_logging:393] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}




Transcribing:   0%|          | 0/1 [00:00<?, ?it/s]

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]

100%|██████████| 1/1 [02:15<00:00, 135.82s/it]


In [27]:
print (data.columns)

Index(['hypothesis', 'reference', 'file_ids', 'hypothesis_clean',
       'reference_clean', 'segment', 'start_offset', 'end_offset', 'start',
       'end'],
      dtype='object')
